# Chapter 15. Text Representations, Embeddings, and Retrieval

## 15.2 Learning Objectives

By the end of this chapter, you should be able to:

- normalize embedding vectors;
- compute cosine similarity between text embeddings;
- perform exact semantic retrieval using matrix multiplication and `torch.topk()`;
- calculate Precision@k, Recall@k, and Mean Reciprocal Rank;
- evaluate a semantic retrieval system using relevance judgments.

## 15.3 Embeddings Revisited: From Categories to Tokens

### 15.3.1 Token IDs and Embeddings

In [4]:
from transformers import AutoTokenizer, AutoModel

repo_id = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModel.from_pretrained(repo_id)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
inputs = tokenizer("The bank approved the loan.", return_tensors="pt")
inputs["input_ids"]

tensor([[ 101, 1996, 2924, 4844, 1996, 5414, 1012,  102]])

### 15.3.2 Input Embeddings

In [6]:
input_embeddings = model.get_input_embeddings()
input_embeddings

Embedding(30522, 384, padding_idx=0)

In [8]:
bank_id = tokenizer.convert_tokens_to_ids("bank")
bank_id

2924

In [14]:
import torch
bank_embedding = input_embeddings(torch.tensor([bank_id]))
bank_embedding[:, :30], bank_embedding.shape

(tensor([[ 0.0492,  0.0312, -0.0702,  0.0593,  0.0814, -0.0581,  0.0271,  0.0161,
           0.0093, -0.0224,  0.0503, -0.0198, -0.0624,  0.1288, -0.0036, -0.0021,
          -0.0082, -0.0012,  0.0651,  0.0523, -0.0068,  0.0270, -0.0084,  0.0737,
          -0.0028,  0.0288,  0.0304, -0.0213, -0.1394, -0.1267]],
        grad_fn=<SliceBackward0>),
 torch.Size([1, 384]))

In [11]:
sentences = [
    "The bank approved the loan.",
    "They sat on the bank of the river.",
]

## 15.4 Contextual Word Embeddings Revisited

In [15]:
inputs = tokenizer(sentences, padding=True, return_tensors="pt")
inputs

{'input_ids': tensor([[ 101, 1996, 2924, 4844, 1996, 5414, 1012,  102,    0,    0,    0],
        [ 101, 2027, 2938, 2006, 1996, 2924, 1997, 1996, 2314, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [16]:
model.eval()

with torch.inference_mode():
    outputs = model(**inputs)

outputs.last_hidden_state.shape

torch.Size([2, 11, 384])

In [19]:
outputs.last_hidden_state[0, 2][:30], outputs.last_hidden_state[1, 5][:30], 

(tensor([ 0.3246,  0.2335, -0.5084, -0.1563, -0.2294, -0.6257,  0.5471,  0.0617,
          0.1467, -0.1831, -0.1169, -0.5345, -0.1759, -0.7436, -0.4449, -0.0546,
         -0.3849, -0.0901,  0.0126,  0.4470, -0.1323, -0.0042, -0.3002, -0.1354,
          0.6542, -0.1733, -0.0711, -0.0205, -0.9364, -0.3025]),
 tensor([ 0.3987,  0.5271, -0.8868, -0.4379, -0.4681,  0.0476,  0.2013, -0.0398,
          0.0220, -0.2248, -0.3866, -0.6498, -0.4500, -0.3506, -0.1162,  0.0352,
         -0.0091,  0.3312,  0.5224,  0.6101,  0.2410, -0.0170, -0.5578, -0.0881,
          0.2137, -0.1553,  0.0062, -0.2102, -1.1757, -0.0813]))

## 15.5 Sentence and Document Embeddings

In [21]:
sentences = [
    "The treatment reduced blood pressure.",
    "The therapy lowered arterial pressure.",
    "The telescope observed a distant galaxy.",
]

In [22]:
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")

with torch.inference_mode():
    outputs = model(**inputs)

token_embeddings = outputs.last_hidden_state
token_embeddings.shape

torch.Size([3, 9, 384])

### 15.5.1 Pooling

In [23]:
token_embeddings.mean(dim=1)

tensor([[-0.2456,  0.8016,  0.3255,  ..., -0.3483, -0.3009,  0.3236],
        [-0.1983,  0.8145,  0.1214,  ...,  0.0705,  0.0719,  0.1404],
        [-0.2250,  0.1222, -0.2998,  ...,  0.0666, -0.4715,  0.1760]])

### 15.5.2 Attention Masks

In [24]:
inputs["attention_mask"]

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])

In [25]:
def mean_pooling(token_embeddings, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(token_embeddings.dtype)

    summed = (token_embeddings * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)

    return summed / counts

In [28]:
sentence_embeddings = mean_pooling(token_embeddings, inputs["attention_mask"])
sentence_embeddings, sentence_embeddings.shape

(tensor([[-0.2504,  0.8186,  0.2809,  ..., -0.3611, -0.3668,  0.3519],
         [-0.1999,  0.8152,  0.0974,  ...,  0.0798,  0.0595,  0.1419],
         [-0.2250,  0.1222, -0.2998,  ...,  0.0666, -0.4715,  0.1760]]),
 torch.Size([3, 384]))

### 15.5.3 Normalizing the Representations

In [29]:
import torch.nn.functional as F

sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)
sentence_embeddings

tensor([[-0.0378,  0.1237,  0.0424,  ..., -0.0546, -0.0554,  0.0532],
        [-0.0302,  0.1234,  0.0147,  ...,  0.0121,  0.0090,  0.0215],
        [-0.0369,  0.0201, -0.0492,  ...,  0.0109, -0.0774,  0.0289]])

In [31]:
sentence_embeddings.norm(dim=1)

tensor([1.0000, 1.0000, 1.0000])

### 15.5.4 A Reusable Encoder

In [32]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
model = model.to(device)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
    

In [33]:
from tqdm import tqdm

def encode_texts(texts, batch_size=64):
    embeddings = []

    for start in tqdm(range(0, len(texts), batch_size)):
        batch = texts[start:start + batch_size]

        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
        inputs = inputs.to(device)

        with torch.inference_mode():
            outputs = model(**inputs)

        batch_embeddings = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
        batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)
        embeddings.append(batch_embeddings.cpu())

    return torch.cat(embeddings, dim=0)

In [36]:
embeddings = encode_texts(sentences)
embeddings, embeddings.shape

100%|█████████████████████████████████████████████| 1/1 [00:00<00:00, 70.79it/s]


(tensor([[-0.0378,  0.1237,  0.0424,  ..., -0.0546, -0.0554,  0.0532],
         [-0.0302,  0.1234,  0.0147,  ...,  0.0121,  0.0090,  0.0215],
         [-0.0369,  0.0201, -0.0492,  ...,  0.0109, -0.0774,  0.0289]]),
 torch.Size([3, 384]))

## 15.6 Normalization and Similarity

We now have numerical representations of complete pieces of text. How should we compare them? A commonly used measure for dense text embeddings is **cosine similarity**. For two vectors (x) and (y):

$$
\Large
\frac{x \cdot y}
{\lVert x\rVert \lVert y\rVert}
$$

Cosine similarity focuses on the direction of the vectors rather than their absolute magnitudes. It ranges from `-1` to `1`, where values closer to `-1` indicate the embeddings likely correspond to sentences with opposite meaning; values closer to `1` indicate the embeddings represent sentences similar in content; and values closer to `0` indicate the sentences are likely unrelated.

### 15.6.1 Computing Similarity

In [37]:
sentences

['The treatment reduced blood pressure.',
 'The therapy lowered arterial pressure.',
 'The telescope observed a distant galaxy.']

In [38]:
F.cosine_similarity(embeddings[0], embeddings[1], dim=0)

tensor(0.7944)

In [39]:
F.cosine_similarity(embeddings[0], embeddings[2], dim=0), F.cosine_similarity(embeddings[1], embeddings[2], dim=0)

(tensor(0.0589), tensor(0.0684))

### 15.6.2 Normalized Embeddings and Dot Products

In [41]:
embeddings.norm(dim=1)

tensor([1.0000, 1.0000, 1.0000])

In [42]:
similarities = (embeddings @ embeddings.T)
similarities

tensor([[1.0000, 0.7944, 0.0589],
        [0.7944, 1.0000, 0.0684],
        [0.0589, 0.0684, 1.0000]])

## 15.7 Semantic Search

Now, let's suppose we have a collection of documents and a query. A traditional search engine may emphasize whether query terms occur in the documents. Dense semantic retrieval takes a different approach, as we represent both query and documents using vectors in the same embedding space and compare those vectors directly (using cosine similarity, for example).

The procedure is:
1. encode the documents;
2. encode the query;
3. calculate similarity scores;
4. rank the documents;
5. return the highest-scoring results.

Let's try it out.

## 15.8 The SciFact Retrieval Dataset

We'll use **SciFact**, one of the scientific fact-checking retrieval datasets included in the BEIR benchmark. BEIR datasets follow a particularly useful retrieval structure:
- a **corpus** of documents;
- **queries**;
- **qrels**, or relevance judgments, associating queries with relevant corpus documents.

We have all the information needed to properly evaluate retrieval.

### 15.8.1 Corpus

In [43]:
from datasets import load_dataset

corpus = load_dataset("BeIR/scifact", "corpus", split="corpus")
corpus

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

corpus/corpus-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.47MB            

corpus/corpus-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating corpus split:   0%|          | 0/5183 [00:00<?, ? examples/s]

Dataset({
    features: ['_id', 'title', 'text'],
    num_rows: 5183
})

In [44]:
corpus[0]

{'_id': '4983',
 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.',
 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, t

In [45]:
def document_text(example):
    title = example["title"].strip()
    text = example["text"].strip()
    if title:
        return f"{title}. {text}"

    return text

In [46]:
documents = [document_text(example) for example in corpus]
document_ids = list(corpus["_id"])

### 15.8.2 Queries

In [48]:
queries = load_dataset("BeIR/scifact", "queries", split="queries")
queries

queries/queries-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 65.0kB            

queries/queries-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating queries split:   0%|          | 0/1109 [00:00<?, ? examples/s]

Dataset({
    features: ['_id', 'title', 'text'],
    num_rows: 1109
})

In [75]:
query_by_id = {example["_id"]: example["text"] for example in queries}
query_by_id['3']

'1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.'

### 15.8.3 Relevance Judgments

In [56]:
qrels = load_dataset("BeIR/scifact-qrels", split="test")
qrels

README.md:   0%|          | 0.00/14.0k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/14.5k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/5.39k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/919 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/339 [00:00<?, ? examples/s]

Dataset({
    features: ['query-id', 'corpus-id', 'score'],
    num_rows: 339
})

In [57]:
from collections import defaultdict

relevant_by_query = defaultdict(set)
for row in qrels:
    if row["score"] > 0:
        relevant_by_query[str(row["query-id"])].add(str(row["corpus-id"]))

## 15.9 Building the Semantic Search System

We have already describe the steps, which are reproduced below:
1. encode the documents;
2. encode the query;
3. calculate similarity scores;
4. rank the documents;
5. return the highest-scoring results.

### 15.9.1 Encode the Documents

In [91]:
corpus_embeddings = encode_texts(documents, batch_size=64)
corpus_embeddings.shape

100%|███████████████████████████████████████████| 81/81 [06:51<00:00,  5.09s/it]


torch.Size([5183, 384])

### 15.9.2 Encode the Query

In [87]:
query_id = '3'
query = query_by_id[query_id]
query

'1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.'

In [89]:
query_embedding = encode_texts([query])
query_embedding.shape

100%|█████████████████████████████████████████████| 1/1 [00:00<00:00, 69.00it/s]


torch.Size([1, 384])

### 15.9.3 Calculate Similarity Scores

In [92]:
scores = (query_embedding @ corpus_embeddings.T)
scores.shape

torch.Size([1, 5183])

## 15.9.4 Rank the Documents

In [93]:
top_scores, top_indices = torch.topk(scores.squeeze(0), k=5)

In [94]:
for rank, (score, index) in enumerate(zip(top_scores, top_indices), start=1):
    index = int(index)

    print(f"Rank: {rank}")
    print(f"Score: {score:.3f}")
    print(f"Document ID: {document_ids[index]}")
    print(corpus[index]["title"])
    print(corpus[index]["text"][:500])
    print()

Rank: 1
Score: 0.651
Document ID: 2739854
Rare and common variants: twenty arguments
Genome-wide association studies have greatly improved our understanding of the genetic basis of disease risk. The fact that they tend not to identify more than a fraction of the specific causal loci has led to divergence of opinion over whether most of the variance is hidden as numerous rare variants of large effect or as common variants of very small effect. Here I review 20 arguments for and against each of these models of the genetic basis of complex traits and conclude that both classes of e

Rank: 2
Score: 0.640
Document ID: 23389795
Common and rare variants in multifactorial susceptibility to common diseases
Here, we give a historical overview of the search for genetic variants that influence the susceptibility of an individual to a chronic disease, from RA Fisher's seminal work to the current excitement of whole-genome association studies (WGAS). We then discuss the concepts behind the identific

In [98]:
relevant_docs = (relevant_by_query[query_id])

for rank, index in enumerate(top_indices, start=1):
    index = int(index)
    doc_id = document_ids[index]
    print(f"Rank {rank} - ID {doc_id} - Match: {doc_id in relevant_docs}")

Rank 1 - ID 2739854 - Match: False
Rank 2 - ID 23389795 - Match: False
Rank 3 - ID 19058822 - Match: False
Rank 4 - ID 15153602 - Match: False
Rank 5 - ID 14717500 - Match: True


### 15.9.5 A Reusable Retrieval Function

In [96]:
import pandas as pd

def retrieve(query, corpus, corpus_embeddings, document_ids, k=5):
    query_embedding = encode_texts([query])
    scores = (query_embedding @ corpus_embeddings.T).squeeze(0)
    k = min(k, len(corpus))
    top_scores, top_indices = torch.topk(scores, k=k)

    results = []
    for rank, (score, index) in enumerate(zip(top_scores, top_indices), start=1):
        index = int(index)
        example = corpus[index]

        results.append({
            "rank": rank,
            "score": float(score),
            "document_id": document_ids[index],
            "title": example["title"],
            "text": example["text"],
        })

    return pd.DataFrame(results)

In [97]:
retrieve(query, corpus, corpus_embeddings, document_ids)

100%|█████████████████████████████████████████████| 1/1 [00:00<00:00, 69.48it/s]


,rank,score,document_id,title,text
0,1,0.651018,2739854,Rare and common variants: twenty arguments,Genome-wide association studies have greatly i...
1,2,0.640482,23389795,Common and rare variants in multifactorial sus...,"Here, we give a historical overview of the sea..."
2,3,0.599482,19058822,A fine-scale map of recombination rates and ho...,"Genetic maps, which document the way in which ..."
3,4,0.597784,15153602,Haplotype-based variant detection from short-r...,The direct detection of haplotypes from short-...
4,5,0.564525,14717500,Rare Variants Create Synthetic Genome-Wide Ass...,Genome-wide association studies (GWAS) have no...


## 15.10 Retrieval Granularity

SciFact documents are relatively short, real-world documents may be much larger, like research papers, books, contracts, manual, or source code. Encoding the entire object using a single vector is unlikely to work well.

Embedding models accept a finite number of input tokens. For example, `all-MiniLM-L6-v2`, for truncates inputs longer than 256 word pieces by default. Anything beyond that limit cannot influence the resulting embedding. If we try encoding a full paper, it's exactly the same as if we were encoding its first 256 tokens only, the remaining information is effectively lost.

Even without a hard input limit, representing a very long document using a single vector creates another problem. Suppose only one paragraph in a 30-page report is relevant to the query. A document-level embedding must somehow summarize all 30 pages. The representation of that one important paragraph may be diluted by everything else and, even if the retrieval system returned the correct document, we would have to search within it for the appropriate paragraph.

### 15.10.1 Chunking

For this reason, long documents are commonly divided into smaller **chunks**. A chunk might correspond to:

- a sentence;
- a paragraph;
- several paragraphs;
- a document section;
- a fixed number of tokens.

Each chunk receives its own embedding. The retrieval unit therefore becomes a passage rather than necessarily the entire source document.
As usual, there is a trade-off when it comes to chunk size. Very small chunks are focused, but may lack context. Very large chunks preserve more context, but may mix unrelated subjects. Some systems also use overlapping chunks so that information near a boundary isn't split awkwardly between them. Unfortunately, there is no universally correct chunk size,  it depends on the documents, the embedding model, and the retrieval task. For SciFact, we'll keep the document-level representation.

## 15.11 Scaling Semantic Retrieval

Our implementation performs an **exact search**. For each query, we compare its embedding against every corpus embedding. At SciFact's scale, this is straightforward, but what happens when the collection contains millions or billions of vectors?

Our current search guarantees that the highest-scoring vectors can be found because every document is compared against the query. This is called **exact nearest-neighbor search**. Its amount of work grows with both the number of vectors and their dimensionality. For modest collections, exact search may remain perfectly practical.

Large vector collections are often searched using **Approximate Nearest Neighbor**, or ANN, indexes. Instead of comparing the query with every candidate, these indexes organize the vector space so that likely neighbors can be found more efficiently. Although this approach is much faster at scale, it may occasionally miss a neighbor.

Technologies such as FAISS, HNSW indexes, pgvector, Qdrant, Chroma, and Milvus provide different approaches or infrastructure for vector search. We won't learn their APIs here. They optimize the same nearest-neighbor problem we've already implemented.

### 15.11.1 Vector Databases

A vector database typically adds capabilities around similarity search, such as:

- persistent storage;
- indexing;
- metadata;
- filtering;
- insertion and deletion;
- distributed search;
- access control.

These features are important when building larger systems. But they are infrastructure around retrieval rather than the source of semantic similarity itself. Our tensor implementation exposes the actual mechanism behind it.

## 15.12 Dense, Lexical, and Hybrid Retrieval

Dense retrieval is powerful, but it isn't universally superior to other search techniques. If you have to search for exact queries such as `CVE-2026-12345` or `TP53 R175H`, or an exact product code, legal citation, model identifier, or rare proper name, literal matching can be extremely valuable.

Traditional retrieval methods operate largely on the terms occurring in the query and documents. Methods such as **BM25** reward documents containing query terms while accounting for properties such as term frequency and how rare those terms are across the collection. Lexical retrieval is particularly useful when exact terminology matters.

Our embedding-based approach is called **dense retrieval** because the texts are represented by dense numerical vectors. Dense retrieval can match related meanings even when the wording differs, as in our example with health- and astronomy-related sentences. The first two sentences expressed closely related ideas despite not using exactly the same words.

Lexical and dense retrieval emphasize different signals, so real systems often combine them. A **hybrid retrieval** system may use both lexical matching and dense semantic similarity, and combine their candidates or scores. Rather than thinking of semantic search as replacing lexical search, it is more useful to think of them as complementary: lexical retrieval emphasizes **what was written**; dense retrieval emphasizes **what was meant**.

## 15.13 Evaluating Retrieval

### 15.13.1 Precision @ K

In [102]:
def precision_at_k(ranking, relevant, k):
    retrieved = set(ranking[:k])
    return len(retrieved & relevant) / k

### 15.13.2 Recall @ K

In [103]:
def recall_at_k(ranking, relevant, k):
    if not relevant:
        return 0.0

    retrieved = set(ranking[:k])

    return len(retrieved & relevant) / len(relevant)

### 15.13.3 Reciprocal Rank

In [104]:
def reciprocal_rank(ranking, relevant):
    for rank, doc_id in enumerate(ranking, start=1):
        if doc_id in relevant:
            return 1.0 / rank

    return 0.0

### 15.13.4 Evaluating the Test Queries

In [105]:
test_query_ids = sorted(relevant_by_query.keys(), key=int)
test_query_texts = [query_by_id[query_id] for query_id in test_query_ids]
query_embeddings = encode_texts(test_query_texts, batch_size=64)

100%|█████████████████████████████████████████████| 5/5 [00:01<00:00,  3.51it/s]


In [106]:
all_scores = (query_embeddings @ corpus_embeddings.T)
all_scores.shape

torch.Size([302, 5183])

In [110]:
max_k = 100
top_indices = torch.topk(all_scores, k=max_k, dim=1).indices
top_indices

tensor([[4132,  766, 5115,  ...,  835, 1150, 1177],
        [ 604, 3143, 2232,  ..., 4963, 3416, 4008],
        [ 506, 3444, 2901,  ..., 2297, 2571, 3001],
        ...,
        [1624,   52,  630,  ..., 1976, 3956, 3468],
        [3513, 1505,  217,  ..., 1814, 3254, 4164],
        [2818, 2925, 2764,  ..., 3556, 4060, 1900]])

Then, for each query ID, we'll assemble a list of the corresponding document IDs:

In [112]:
rankings = {}

for query_id, indices in zip(test_query_ids, top_indices):
    rankings[query_id] = [document_ids[int(index)] for index in indices]

In [114]:
print(rankings['3'])

['2739854', '23389795', '19058822', '15153602', '14717500', '32181055', '4378885', '4632921', '1388704', '6078882', '13914198', '1544804', '1067605', '45200347', '4414547', '13916484', '3662132', '13519661', '25536577', '15570962', '24144677', '13949015', '4457160', '10944947', '36713289', '17088791', '11532028', '52850476', '14464451', '2485101', '2107238', '3444507', '12271486', '14717213', '2359152', '9704467', '7324039', '6535392', '24896957', '41782935', '2454002', '19293654', '810480', '1286352', '10766688', '461550', '32770503', '84784389', '9872468', '2402323', '13791788', '7838799', '13123189', '35861290', '27158570', '16172576', '4465608', '10145528', '380526', '12030680', '26996935', '2095573', '4421746', '5650232', '4489217', '11117679', '4325398', '6503534', '18236313', '7059897', '12428137', '9650982', '44409062', '13618987', '7211056', '15617300', '15155862', '54561709', '11728637', '22067786', '1265945', '175735', '25571386', '20280410', '36082224', '2722988', '6532806'

The helper function below computes the metrics (precision@k, recall@k, and mean reciprocal rank), given the rankings, the ground truth (`qrels`), and some values of K of interest.

In [116]:
def evaluate_retrieval(rankings, relevant_by_query, ks=(1, 5, 10, 100),):
    results = {}

    for k in ks:
        precisions = []
        recalls = []

        for query_id, ranking in rankings.items():
            relevant = relevant_by_query[query_id]
            precisions.append(precision_at_k(ranking, relevant, k))
            recalls.append(recall_at_k(ranking, relevant, k))

        results[f"Precision@{k}"] = sum(precisions) / len(precisions)
        results[f"Recall@{k}"] = sum(recalls) / len(recalls)

    reciprocal_ranks = [reciprocal_rank(rankings[query_id], relevant_by_query[query_id]) 
                        for query_id in rankings]
    results["MRR@100"] = sum(reciprocal_ranks) / len(reciprocal_ranks)
    return results

Let's see how well our system performed:

In [117]:
metrics = evaluate_retrieval(rankings, relevant_by_query)
pd.Series(metrics)

Precision@1      0.503311
Recall@1         0.487307
Precision@5      0.164901
Recall@5         0.748510
Precision@10     0.088079
Recall@10        0.784492
Precision@100    0.010563
Recall@100       0.928808
MRR@100          0.612942
dtype: float64

We can now compare retrieval configurations quantitatively instead of deciding whether they "look good" based on a handful of examples.

For instance, we could change:
- the embedding model;
- whether document titles are included or not;
- the maximum input length (respecting the model's maximum length);
- the representation of documents;
- the retrieval method;

and determine whether retrieval improves or deteriorates.

### 15.13.5 Interpreting Retrieval Metrics

Different metrics emphasize different properties of the ranking:
- `Precision@k` emphasizes how much of the retrieved set is useful;
- `Recall@k` emphasizes whether the system found the relevant documents at all;
- MRR emphasizes how early the first useful result appears.

The appropriate metric depends on the application. A search interface showing only three results may care strongly about precision near the top, while a system collecting evidence for further analysis may instead prioritize high recall. Information retrieval benchmarks also commonly use ranking-sensitive metrics such as **nDCG**, which gives greater weight to relevant documents appearing near the top of the ranking and can accommodate graded relevance judgments. We won't cover every retrieval metric here.

## 15.14 Where Retrieval Is Used

Semantic retrieval is useful in many settings, including:

- scientific evidence retrieval;
- semantic document search;
- finding related support tickets;
- matching questions with existing answers;
- duplicate detection;
- recommendation;
- code search;
- finding related products or records.

Retrieval may also be used as one component of systems that combine search with generative models. But retrieval is a complete problem in its own right.

## 15.15 Looking Ahead

Let's review the progression we've followed. We first encountered embeddings in Chapter 3 as learned representations of categorical values. In this chapter, the same basic mechanism appeared at the entrance of a language model: token IDs were used to retrieve learned **input embeddings**. The Transformer then turned those vectors into **contextual token embeddings**, whose representations depended on the surrounding text. Pooling allowed us to combine those contextual representations into one vector representing a complete sentence or document. Normalization made those vectors convenient to compare using cosine similarity—or, equivalently for unit vectors, a dot product. Once documents and queries occupied the same embedding space, semantic retrieval became a nearest-neighbor problem.

For SciFact, the core search operation was simply a dot-product followed by a top-K call. We then evaluated those rankings against real relevance judgments. The models we've used so far have been particularly useful because of the **representations** they produce. 

Modern large language models solve another problem. They generate text by repeatedly answering a deceptively simple question: given the tokens I've seen so far, which token should come next? To understand how that works, we need to examine the Transformer architecture from a different direction. In the next chapter, we'll explore **self-attention**, **causal masking**, **decoder-based Transformers**, and **autoregressive language modeling**—the foundations of modern generative LLMs.